In [7]:
from urllib.request import urlopen
from bs4 import BeautifulSoup

html = urlopen('http://en.wikipedia.org/wiki/Kevin_Bacon')
bs = BeautifulSoup(html, 'html.parser')

x = 0
for link in bs.find_all('a'): # 'a' is a tag
    if 'href' in link.attrs:
        print(link.attrs['href'])
        # added this to limit the link returns to 5
        x += 1
        if x == 5: break
   

#bodyContent
/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random


#### Retrieving Articles Only

In [14]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import re

html = urlopen('http://en.wikipedia.org/wiki/Kevin_Bacon')
bs = BeautifulSoup(html, 'html.parser')

x = 0
for link in bs.find('div', {'id':'bodyContent'}).find_all(
    'a', href=re.compile(r'^(/wiki/)((?!:).)*$')):
    print(link.attrs['href'])
    # added this to limit the link returns to 5
    x+=1
    if x == 5: break


/wiki/Kevin_Bacon_(disambiguation)
/wiki/Tribeca_Festival
/wiki/Philadelphia
/wiki/Kevin_Bacon_filmography
/wiki/Kyra_Sedgwick


#### Random Walk

In [34]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import datetime
import random
import re

# Changed for windows error in book
random.seed(int(datetime.datetime.now().timestamp()))

print(int(datetime.datetime.now().timestamp()))

def getLinks(articleUrl):
    html = urlopen(f'http://en.wikipedia.org{articleUrl}')
    bs = BeautifulSoup(html, 'html.parser')
    return bs.find('div',
                   {'id':'bodyContent'}).find_all(
                   'a', href=re.compile(r'^(/wiki/)((?!:).)*$'))
links = getLinks('/wiki/Kevin_Bacon')

x = 0
while len(links) > 0:
    newArticle = links[random.randint(0,len(links)-1)].attrs['href']
    print(newArticle)
    links = getLinks(newArticle)
    # added this to limit the link returns to 5
    x+=1 
    if x>5: break

        

1754019145
/wiki/Stanley_Tucci
/wiki/51st_Telluride_Film_Festival
/wiki/Telluride_Film_Festival
/wiki/Salman_Rushdie
/wiki/Republican_Party_(United_States)
/wiki/Russia


#### Recursively crawling an entire site

In [41]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import re

pages = set()
def getLinks(pageUrl):
    html = urlopen(f'http://en.wikipedia.org{pageUrl}')
    bs = BeautifulSoup(html, 'html.parser')
    for link in bs.find_all('a', href=re.compile(r'^(/wiki/)')):
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                # add new page to set
                newPage = link.attrs['href']
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)

getLinks('/wiki/Kevin_Bacon')

/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About
/wiki/Help:Contents
/wiki/Help:Introduction
/wiki/Wikipedia:Community_portal
/wiki/Special:RecentChanges
/wiki/Wikipedia:File_upload_wizard
/wiki/Special:SpecialPages
/wiki/Special:Search
/wiki/Special:MyContributions
/wiki/Special:MyTalk
/wiki/Special:WhatLinksHere/User_talk:2600:1702:5B81:79B0:9D42:EB54:1E17:9843


HTTPError: HTTP Error 404: Not Found

#### Collecting Data Across an Entire Site

In [47]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import re

pages = set()
def getLinks(pageUrl):
    html = urlopen(f'http://en.wikipedia.org{pageUrl}')
    bs = BeautifulSoup(html, 'html.parser')
    try:
        print(bs.h1.get_text())
        bodyContent = bs.find('div', {'id':'bodyContent'}).find_all('p')
        if len(bodyContent):
            print(bodyContent[0])
        print(bs.find(id='ca-edit').find('a').attrs['href'])
    except AttributeError:
        print('This page is missing something! Continuing.')

    x = 0
    for link in bs.find_all('a', href=re.compile(r'^(/wiki/)')):
        # added to limit the number of returns
        x +=1
        if x == 5: break
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                # add new page to set
                newPage = link.attrs['href']
                print('-'*20)
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)

getLinks('/wiki/General-purpose_programming_language')

General-purpose programming language
<p>In <a class="mw-redirect" href="/wiki/Computer_software" title="Computer software">computer software</a>, a <b>general-purpose programming language</b> (<b>GPL</b>) is a <a href="/wiki/Programming_language" title="Programming language">programming language</a> for building <a href="/wiki/Software" title="Software">software</a> in a wide variety of application <a href="/wiki/Domain_(software_engineering)" title="Domain (software engineering)">domains</a>. Conversely, a <a href="/wiki/Domain-specific_language" title="Domain-specific language">domain-specific programming language</a> (DSL) is used within a specific area. For example, <a href="/wiki/Python_(programming_language)" title="Python (programming language)">Python</a> is a GPL, while <a href="/wiki/SQL" title="SQL">SQL</a> is a DSL for <a href="/wiki/Query_language" title="Query language">querying relational databases</a>.
</p>
/w/index.php?title=General-purpose_programming_language&action=

#### Crawling Across the Internet

In [49]:
from urllib.request import urlopen
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import re
import datetime
import random

In [64]:
# Retrieves a list of all internal links found on a page
def getInternalLinks(bs, url):
    # extract the network location (website or IP, port, credentials)
    netloc = urlparse(url).netloc 

    # Extract the network scheme (http / https)
    scheme = urlparse(url).scheme
    
    internalLinks = set()

    # Cycle through all the <a> anchor tags
    for link in bs.find_all('a'):
        
        # Check to see if the anchor tag contains an href attribute
        if not link.attrs.get('href'):
            continue

        # Retrieve the key 'href' value from the attrs Tag dictionary 
        parsed = urlparse(link.attrs['href'])

        # Check to see if the 'href' value contains a network location
        # The 'ref' value is normally a Url
        # If it doesn't have a network location, build the full Url
        # and add it to the internalLinks set
        if parsed.netloc =='':
            internalLinks.add(f'{scheme}://{netloc}/{link.attrs["href"].strip("/")}')
        
        # Else... check if the 'href' Url network location matches the 
        # original website network location. If it does match, add the 
        # Url to the internalLink set 
        elif parsed.netloc == netloc:
            internalLinks.add(link.attrs['href'])
    return list(internalLinks)

In [65]:
# Retrieves a list of all external links found on a page
def getExternalLinks(bs, url):
    netloc = urlparse(url).netloc
    externalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc != '' and parsed.netloc != netloc:
            externalLinks.add(link.attrs['href'])
    return list(externalLinks)

In [66]:
# Retrieve a random external link found on a website
def getRandomExternalLink(startingPage):
    bs = BeautifulSoup(urlopen(startingPage), 'html.parser')
    externalLinks = getExternalLinks(bs, startingPage)
    if not len(externalLinks):
        print('No external links, looking around the site for one')
        internalLinks = getInternalLinks(bs, startingPage)
        return getRandomExternalLink(random.choice(internalLinks))
    else:
        return random.choice(externalLinks)

In [67]:
# Follow random external links found on a website
def followExternalOnly(startingSite):
    externalLink = getRandomExternalLink(startingSite)
    print(f'Random external link is: {externalLink}')
    followExternalOnly(externalLink)

In [68]:
followExternalOnly('https://www.oreilly.com/')

Random external link is: https://learning.oreilly.com/search/?query=author%3A%22Ken%20Kousen%22&extended_publisher_data=true&highlight=true&include_assessments=false&include_case_studies=true&include_courses=true&include_playlists=true&include_collections=true&include_notebooks=true&include_sandboxes=true&include_scenarios=true&is_academic_institution_account=false&source=user&sort=date_added&facet_json=true&json_facets=true&page=0&include_facets=false
Random external link is: https://www.oreilly.com/online-learning/pricing.html
Random external link is: https://www.linkedin.com/company/oreilly-media


HTTPError: HTTP Error 999: Request denied

#### Collect all External Links from a Site

In [69]:
def getAllExternalLinks(url):
    bs = BeautifulSoup(urlopen(url), 'html.parser')
    internalLinks = getInternalLinks(bs, url)
    externalLinks = getExternalLinks(bs, url)
    for link in externalLinks:
        if link not in allExtLinks:
            allExtLinks.append(link)
            print(link)

    for link in internalLinks:
        if link not in allIntLinks:
            allIntLinks.append(link)
            getAllExternalLinks(link)

In [70]:
allExtLinks = []
allIntLinks = []
allIntLinks.append('https://oreilly.com')
getAllExternalLinks('https://www.oreilly.com')             

https://www.youtube.com/user/OreillyMedia
https://channelstore.roku.com/details/c9d25fa651f0ad84e484b0dfd4b20172:856a240ad268961983e91ae52c1e1e5c/oreilly
https://www.amazon.com/OReilly-Media-Inc/dp/B087YYHL5C/ref=sr_1_2?dchild=1&keywords=oreilly&qid=1604964116&s=mobile-apps&sr=1-2
https://learning.oreilly.com/search/?query=author%3A%22Neal%20Ford%22&extended_publisher_data=true&highlight=true&include_assessments=false&include_case_studies=true&include_courses=true&include_playlists=true&include_collections=true&include_notebooks=true&include_sandboxes=true&include_scenarios=true&is_academic_institution_account=false&source=user&sort=date_added&facet_json=true&json_facets=true&page=0&include_facets=false
https://learning.oreilly.com/search/?query=author%3A%22Sari%20Greene%22&extended_publisher_data=true&highlight=true&include_assessments=false&include_case_studies=true&include_courses=true&include_playlists=true&include_collections=true&include_notebooks=true&include_sandboxes=true&incl

ValueError: unknown url type: '//www.oreilly.com/animals.csp'